In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, types
from sqlalchemy import text 

In [14]:
from dotenv import dotenv_values

config = dotenv_values()

pg_user = config['POSTGRES_USER'] 
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

In [15]:
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

In [16]:
engine = create_engine(url, echo=False)

In [17]:
my_schema = 'team_1' 

with engine.begin() as conn: 
    result = conn.execute(text(f'SET search_path TO {my_schema};'))

In [18]:
df_hypertension = pd.read_sql(sql=text("""SELECT 
    occupation, 
    gender,
    COUNT(*) AS num_persons,
    AVG(stress_level) AS avg_stress_level,
    AVG(blood_pressure_systolic) AS avg_bp_systolic,
    AVG(blood_pressure_diastolic) AS avg_bp_diastolic,
    100.0 * SUM(CASE 
                  WHEN blood_pressure_systolic >= 140 OR blood_pressure_diastolic >= 90 
                  THEN 1 ELSE 0 
                END) / COUNT(*) AS hypertension_rate_percent
FROM 
    sleep_health_lifestyle
GROUP BY 
    occupation, gender
HAVING 
    COUNT(*) >= 20
ORDER BY 
    num_persons DESC;"""), con=engine)
df_hypertension

,occupation,gender,num_persons,avg_stress_level,avg_bp_systolic,avg_bp_diastolic,hypertension_rate_percent
0,Nurse,Female,73,5.547945,138.520548,93.726027,89.041096
1,Doctor,Male,69,6.840580,123.144928,80.666667,5.797101
2,Lawyer,Male,45,5.044444,129.888889,84.933333,0.000000
3,Accountant,Female,36,4.555556,117.722222,76.944444,0.000000
4,Teacher,Female,35,4.285714,131.285714,87.142857,77.142857
5,Engineer,Female,32,3.000000,125.000000,80.000000,0.000000
6,Salesperson,Male,32,7.000000,130.000000,85.000000,0.000000
7,Engineer,Male,31,4.806452,126.838710,82.806452,0.000000


In [19]:
df_active_sedentary = pd.read_sql(sql=text("""SELECT 
  CASE 
    WHEN occupation IN ('Nurse', 'Doctor', 'Teacher', 'Engineer') THEN 'Active'
    WHEN occupation IN ('Lawyer', 'Accountant', 'Salesperson') THEN 'Sedentary'
    ELSE 'Other'
  END AS job_type,
  occupation,
  gender,
  -- age_group,
  COUNT(*) AS num_persons,
  AVG(quality_of_sleep) AS avg_quality_of_sleep,
  AVG(sleep_duration) AS avg_sleep_duration
FROM 
  SLEEP_HEALTH_LIFESTYLE
WHERE 
  occupation IN ('Nurse', 'Doctor', 'Teacher', 'Engineer', 
                 'Lawyer', 'Accountant', 'Salesperson')
GROUP BY 
  job_type, occupation, gender --, age_group
HAVING 
  COUNT(*) >= 10;
"""), con=engine)
df_active_sedentary

,job_type,occupation,gender,num_persons,avg_quality_of_sleep,avg_sleep_duration
0,Sedentary,Salesperson,Male,32,6.000000,6.403125
1,Active,Nurse,Female,73,7.369863,7.063014
2,Active,Teacher,Female,35,7.114286,6.705714
3,Sedentary,Accountant,Female,36,7.888889,7.111111
4,Active,Engineer,Female,32,9.000000,8.425000
5,Sedentary,Lawyer,Male,45,7.933333,7.422222
6,Active,Engineer,Male,31,7.806452,7.535484
7,Active,Doctor,Male,69,6.579710,6.934783


In [20]:
df_active_sedentary.describe()

,num_persons,avg_quality_of_sleep,avg_sleep_duration
count,8.000000,8.000000,8.000000
mean,44.125000,7.461567,7.200057
std,17.191672,0.920695,0.613991
min,31.000000,6.000000,6.403125
25%,32.000000,6.980642,6.877516
50%,35.500000,7.588157,7.087062
75%,51.000000,7.900000,7.450538
max,73.000000,9.000000,8.425000
